<a href="https://colab.research.google.com/github/astronerd21/Deep-Learning-Oil-Slick-Detection/blob/12-preprocessing-filter-out-image-chips-with-excessive-nodata-50/OilSlick_Experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!pip install rasterio captum

In [ ]:
import os
import sys
from getpass import getpass

github_token = getpass('Füge hier deinen GitHub Personal Access Token ein (ghp_...): ')

repo_url = f"https://{github_token}@github.com/astronerd21/Deep-Learning-Oil-Slick-Detection.git"

%cd /content
!rm -rf Deep-Learning-Oil-Slick-Detection

!git clone -b 12-preprocessing-filter-out-image-chips-with-excessive-nodata-50 {repo_url}

sys.path.append('/content/Deep-Learning-Oil-Slick-Detection')

try:
    from src.dataset import SARDataset
    from src.model import SARResNet
    print("Erfolg! GitHub Code und Module sind bereit.")
except ImportError as e:
    print(f"Fehler beim Laden: {e}.")

In [ ]:
import os

drive_dir = '/content/drive/MyDrive/OilSlickProject/data/OilSlick/'
local_temp_dir = '/content/temp_archives/'
local_base_dir = '/content/data/'
local_images_dir = '/content/data/images_s1'

os.makedirs(local_temp_dir, exist_ok=True)
os.makedirs(local_base_dir, exist_ok=True)
os.makedirs(local_images_dir, exist_ok=True)

# Prüfen, ob die 981 Bilder schon da sind
anzahl_bilder = len(os.listdir(local_images_dir))

if anzahl_bilder < 980:
    print(" Schritt 1/3: Kopiere 7.5 GB Archive lokal in den Speicher")
    !cp {drive_dir}OilSlick-images_s1-00.tar {local_temp_dir}
    !cp {drive_dir}OilSlick-images_s1-01.tar {local_temp_dir}

    print(" Schritt 2/3: Entpacke Bilder.")
    !tar -xf {local_temp_dir}OilSlick-images_s1-00.tar -C {local_base_dir}
    !tar -xf {local_temp_dir}OilSlick-images_s1-01.tar -C {local_base_dir}

    print(" Schritt 3/3: Räume temporäre Dateien auf.")
    !rm -rf {local_temp_dir}

    print("\n Setup komplett!")
else:
    print("Bilder sind bereits entpackt!")

print(f"Aktuelle Anzahl der S1-Bilder: {len(os.listdir(local_images_dir))}")

In [ ]:
import os
import random
import rasterio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Define paths
img_dir = '/content/data/images_s1/'
meta_path = '/content/drive/MyDrive/OilSlickProject/data/OilSlick/metadata.csv'

# 2. Select 5 random images
all_files = [f for f in os.listdir(img_dir) if f.endswith('.tif')]
sample_files = random.sample(all_files, 5)

# Helper function to normalize radar contrast
def stretch_contrast(band):
    # Ignores invalid pixels and calculates the 2nd and 98th percentiles
    vmin, vmax = np.percentile(band[~np.isnan(band)], (2, 98))
    # Everything outside these limits is clipped
    return np.clip(band, vmin, vmax)

# 3. Build the gallery
fig, axes = plt.subplots(2, 5, figsize=(20, 8))

for i, file_name in enumerate(sample_files):
    sample_id = file_name.replace('_s1.tif', '')

    # Derive label robustly from the filename
    if sample_id.startswith('pos') or sample_id.startswith('ext_pos'):
        is_oil = "OIL SLICK (1)"
    elif sample_id.startswith('neg') or sample_id.startswith('ext_neg'):
        is_oil = "NO OIL (0)"
    else:
        is_oil = "Unknown"

    img_path = os.path.join(img_dir, file_name)

    with rasterio.open(img_path) as src:
        vv = src.read(1)
        vh = src.read(2)

    # Apply contrast correction
    vv_vis = stretch_contrast(vv)
    vh_vis = stretch_contrast(vh)

    # Top row: VV Channel
    ax_vv = axes[0, i]
    ax_vv.imshow(vv_vis, cmap='gray')
    ax_vv.set_title(f"VV Channel | {sample_id}\nLabel: {is_oil}", fontsize=11)
    ax_vv.axis('off')

    # Bottom row: VH Channel
    ax_vh = axes[1, i]
    ax_vh.imshow(vh_vis, cmap='gray')
    ax_vh.set_title(f"VH Channel | {sample_id}\nLabel: {is_oil}", fontsize=11)
    ax_vh.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
%cd /content/Deep-Learning-Oil-Slick-Detection

print("=== Starte präzise Datenbereinigung ===")
!PYTHONPATH=. python3 -m src.clean_dataset \
    --data-dir "/content/data/images_s1" \
    --splits-in-dir "/content/drive/MyDrive/OilSlickProject/data/OilSlick/splits/random"

In [ ]:
%cd /content/Deep-Learning-Oil-Slick-Detection

print("=== Starte Modelltraining mit den bereinigten Splits ===")
!PYTHONPATH=. python3 -m src.train \
    --data-dir "/content/data/images_s1" \
    --train-split "/content/cleaned_splits/train_clean.txt" \
    --val-split-file "/content/cleaned_splits/val_clean.txt" \
    --epochs 20 \
    --batch-size 16 \
    --output "/content/drive/MyDrive/OilSlickProject/data/OilSlick/checkpoints/best_cleaned_model.pt"